### GIAI ĐOẠN 1: CV Text Extractor
Xử lý CV theo 3 trường hợp:
- Mức 1: PDF text-based
- Mức 2: PDF multii-column / complex layout
- Mức 3: PDF scan/ image-based

#### 1.  Imports thư viện

Khai báo và nạp tất cả các thư viện Python cần thiết để xử lý PDF, xử lý chuỗi, hình ảnh, OCR và dữ liệu

In [1]:
import fitz
import re
import unicodedata
from collections import Counter
from pathlib import Path
from typing import Any, Optional

import io

from PIL import Image, ImageFilter, ImageOps
import matplotlib.pyplot as plt
import pytesseract
from pytesseract import Output
import pandas as pd
import numpy as np
from pprint import pprint

#### 2. Hằng số, regrex, cấu hình

Thiết lập các hằng số cho OCR, cấu hình đường dẫn thư mục, định nghĩa các từ khóa cho từng phần của CV và các biểu thức chính quy để trích xuất thông tin

In [23]:
OCR_LANG = "vie+eng"
OCR_RENDER_DPI = 400
OCR_LAYOUT_PSM = 4
OCR_MIN_CONFIDENCE = 0.3
OCR_MIN_TEXT_LENGTH = 40

pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "process_AI" else CURRENT_DIR
DATA_DIR = PROJECT_ROOT / "data"
DEFAULT_CV_DIR = PROJECT_ROOT / "media" / "cvs"
MIN_TEXT_LENGTH_WARNING = 300

#Từ điển chứa các từ khóa đồng nghĩa nhận diện tiêu đề của các mục trong CV
SECTION_ALIASES = {
    "personal_info": [
        "thong tin ca nhan", "lien he", "contact", "personal information"
    ],
    "skills": [
        "ky nang", "skills", "skill"
    ],
    "certificates": [
        "chung chi", "certificates", "certifications"
    ],
    "objective": [
        "muc tieu nghe nghiep", "objective", "career objective", "summary", "profile"
    ],
    "education": [
        "hoc van", "education", "academic background"
    ],
    "experience": [
        "kinh nghiem lam viec", "kinh nghiem", "work experience", "experience", "employment history"
    ],
    "activities": [
        "hoat dong", "activities"
    ],
    "projects": [
        "du an", "projects"
    ],
    "languages": [
        "ngon ngu", "languages"
    ],
}

#Tên tiêu đề chuẩn hóa được dùng khi xuất kết quả
SECTION_TITLES = {
    "header": "THONG TIN DAU CV",
    "personal_info": "THONG TIN CA NHAN",
    "objective": "MUC TIEU NGHE NGHIEP",
    "education": "HOC VAN",
    "experience": "KINH NGHIEM LAM VIEC",
    "skills": "KY NANG",
    "certificates": "CHUNG CHI",
    "activities": "HOAT DONG",
    "projects": "DU AN",
    "languages": "NGON NGU",
    "other": "KHAC",
}

#Danh sách xác định thứ tự sắp xếp các mục khi xuất thành văn bản
PREFERRED_SECTION_ORDER = [
    "header",
    "personal_info",
    "objective",
    "education",
    "experience",
    "skills",
    "certificates",
    "activities",
    "projects",
    "languages",
    "other",
]

YEAR_RANGE_RE = re.compile(
    r"\b((?:19|20)\d{2})\s*[-\u2013\u2014]\s*((?:19|20)\d{2}|present|current|nay|hien tai)\b",
    flags=re.IGNORECASE
)

DEGREE_PATTERNS = [
    (r"\b(ph\.?d|doctor(?:ate)?|tien si)\b", "phd"),
    (r"\b(master|m\.?sc|mba|thac si)\b", "master"),
    (r"\b(bachelor|b\.?sc|b\.?eng|cu nhan|dai hoc)\b", "bachelor"),
    (r"\b(college|cao dang)\b", "college"),
    (r"\b(diploma|trung cap)\b", "diploma"),
]

SCHOOL_HINT_RE = re.compile(
    r"\b(truong|tr??ng|university|college|hoc vien|h?c vi?n|institute)\b",
    flags=re.IGNORECASE,
)

MAJOR_REGEX = re.compile(
    r"\b(?:in|major in|specialized in|specialization in)\s+([A-Za-z][A-Za-z\s&/\-]{2,60})",
    flags=re.IGNORECASE,
)

#### 3. Text helper

Cung cấp các hàm tiện ích để làm sạch văn bản thô, xóa dấu tiếng việt và chuẩn hóa các tiêu đề
- normalize_pdf_text: Hàm thay thế các ký tự null, khoảng trắng thừa và chuẩn hóa dấu xuống dòng trong text PDF
- remove_vietnamese_accents: Hàm phân tích Unicode (NFD), loại bỏ dấu thanh và gộp lại NFC để xóa dấu tiếng việt
- normalize_for_heading: Hàm đưa chuỗi về chữ thường, xóa dấu, loại bỏ ký tự đặc biệc để dễ so sánh tiêu đề
- normalized_section_aliases: Tiền xử lý tập hợp các từ khóa alias bằng hàm normalize_for_heading
- detect_section_heading: Kiểm tra xem text chuẩn hóa có nằm trong danh sách tiêu đề CV đã cấu hình không
- clean_line_text: Hàm xóa các biểu tương icon, chuẩn hóa gạch đầu dòng và khoảng trắng trên một dòng
- clean_line_text_and_lower: Làm sạch và đưa một thuật ngữ về chữ thường

In [ ]:
#Hàm làm sạch văn bản thô được trích xuất từ file PDF. 
def normalize_pdf_text(text: str) -> str:
    if not text:
        return ""

    text = text.replace("\x00", " ") #x00: kí tự null, \u00a0: Khoảng trắng đặc biệc
    text = text.replace("\u00a0", " ")
    text = re.sub(r"\r\n|\r", "\n", text) #Chuẩn hóa xuống dòng kiểu Windows \r\n

   
    lines = [re.sub(r"[ \t]+", " ", line).strip() for line in text.split("\n")]

    cleaned_lines = []
    prev_empty = False
    for line in lines:
        if not line:
            if not prev_empty:
                cleaned_lines.append("")
            prev_empty = True
        else:
            cleaned_lines.append(line)
            prev_empty = False

    return "\n".join(cleaned_lines).strip()

#Chuyển đổi kí dự tiếng việt có dấu thành không dấu
def remove_vietnamese_accents(text: str) -> str:
    if not text:
        return ""

    text = text.replace("\u0111", "d").replace("\u0110", "D")
    text = unicodedata.normalize("NFD", text)
    text = "".join(ch for ch in text if unicodedata.category(ch) != "Mn")
    return unicodedata.normalize("NFC", text)

#Chuẩn hóa 1 chuỗi để so vưới tiêu đề
def normalize_for_heading(text: str) -> str:
    if not text:
        return ""

    text = unicodedata.normalize("NFC", text)
    text = remove_vietnamese_accents(text).lower()
    text = re.sub(r"[\ue000-\uf8ff]", " ", text)
    text = re.sub(r"[^a-z0-9\s&/+-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

#Tiền xử lý các tiêu đề phụ để so sánh với các tiêu đề đã biết
NORMALIZED_SECTION_ALIASES = {
    section: {normalize_for_heading(alias) for alias in aliases}
    for section, aliases in SECTION_ALIASES.items()
}

#Nhận diện 1 dòng có phải tiêu đề không
def detect_section_heading(text: str) -> Optional[str]:
    norm = normalize_for_heading(text)

    if not norm:
        return None

    for section, aliases in NORMALIZED_SECTION_ALIASES.items():
        if norm in aliases:
            return section

    return None

#Làm sạch chi tiết trên một dòng đơn lẻ
def clean_line_text(text: str) -> str:
    if not text:
        return ""

    text = unicodedata.normalize("NFC", text)
    text = text.replace("\x00", " ")
    text = text.replace("\u00a0", " ")
    text = re.sub(r"[\ue000-\uf8ff]", " ", text)
    text = re.sub(r"[\U0001F4C5\U0001F464\U00002709\U0000260E\U0001F4DE\U0001F4CD\U0001F517\U0001F310\U0001F3E0\U0001F382]", " ", text)
    text = re.sub(r"[\u2022\u25cf\u25aa\u25a0\u25c6\u25b6\u25ba]", "-", text)
    text = text.replace("\u2013", "-").replace("\u2014", "-")
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()

#Làm sạch toàn bộ văn bản CV, giữ nguyên cấu trúc dòng và đoạn
def clean_cv_text(text: str) -> str:
    if not text:
        return ""

    text = unicodedata.normalize("NFC", text)
    text = text.replace("\x00", " ")
    text = text.replace("\u00a0", " ")
    text = re.sub(r"\r\n|\r", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)

    lines = [line.strip() for line in text.split("\n")]

    cleaned_lines = []
    prev_empty = False
    for line in lines:
        if not line:
            if not prev_empty:
                cleaned_lines.append("")
            prev_empty = True
        else:
            cleaned_lines.append(line)
            prev_empty = False

    text = "\n".join(cleaned_lines)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

#Chuẩn hóa thuật ngữ
def clean_line_text_and_lower(term: str) -> str:
    term = clean_line_text(str(term)).strip().lower()
    term = re.sub(r"\s+", " ", term)
    return term

Các hàm thao tác trên chuỗi, loại bỏ trùng lặp và xây dựng cấu trúc kết quả lưu trữ
- compact_text: Rút gọn chuỗi dài thành chuỗi ngắn, thay vì hiển thị tất cả quá dài thì nó sẽ ... để rút gọn lại
- dedupe_lines_keep_order: dùng cho các dòng văn bản, đặc biệt là tiêu đề/heading/section title. Nó xóa trùng nhưng giữ nguyên dòng gốc đầu tiên.
- unique_keep_order: dùng cho danh sách item đã trích xuất như kỹ năng, email, số điện thoại, keyword, term. Nó xóa trùng và trả về bản đã làm sạch + chữ thường.
- build_empty_sections: Khởi tạo từ điển chứa các danh sách rỗng tương ứng với từng phần cv
- build_extraction_result: Hàm này tạo một dictionary kết quả chuẩn cho quá trình trích xuất PDF/CV.

In [4]:
#Rút gọn văn bản dài để hiển thị trong debug hoặc cảnh báo
def compact_text(text: str, max_len: int = 180) -> str:
    if not text:
        return ""
    text = str(text).strip().replace("\n", " ")
    text = re.sub(r"\s+", " ", text)
    if len(text) <= max_len:
        return text
    return text[:max_len] + "..."

#Loại bỏ các dòng trùng lặp trong một danh sách, giữ nguyên thứ tự xuất hiện
def dedupe_lines_keep_order(lines: list[str]) -> list[str]: 
    result = []
    seen = set()

    for line in lines:
        key = normalize_for_heading(line)

        if not key:
            continue

        if key in seen:
            continue

        seen.add(key)
        result.append(line)

    return result

#Loại bỏ các mục trùng lặp trong một danh sách, giữ nguyên thứ tự xuất hiện
def unique_keep_order(items: list[str]) -> list[str]:
    result = []
    seen = set()

    for item in items:
        value = clean_line_text_and_lower(item)
        if not value:
            continue
        if value in seen:
            continue
        seen.add(value)
        result.append(value)

    return result


def build_empty_sections()-> dict[str, list[str]]:
    return {section: [] for section in PREFERRED_SECTION_ORDER}


def build_extraction_result(pdf_type: str,
    method: str,
    text: str = "",
    warnings: list[str] | None = None,
) -> dict[str, Any]:
    return {
        "text": text,
        "sections": build_empty_sections(),
        "pdf_type": pdf_type,
        "method": method,
        "warnings": list(warnings or []),
        "layout_debug": [],
    }

### 4. Xử lý PDF

##### 4.1 PDF type detection

Kiểm tra đặc điểm của file PDF để xác định phương pháp trích xuất phù hợp nhất:
- is_probably_scanned_pdf: Kiểm tra PDF có phải là PDF scan hay không
- is_probably_multi_column: Kiểm tra xem PDF có khả năng là bố cục nhiều cột không

In [5]:
#Nhận diện xem file PDF có phải là dạng scan hay không dựa trên số lượng ký tự trung bình trên mỗi trang
def is_probably_scanned_pdf(pdf_path: str, min_chars_per_page: int = 80) -> bool:
    doc = fitz.open(str(pdf_path))

    try:
        if len(doc) == 0:
            return True

        total_chars = 0
        for page in doc:
            text = page.get_text("text") or ""
            total_chars += len(text.strip())

        avg_chars = total_chars / len(doc)
        return avg_chars < min_chars_per_page
    finally:
        doc.close()

#Nhận diện xem file PDF có phải là dạng nhiều cột hay không dựa trên độ phân tán của vị trí các khối văn bản
def is_probably_multi_column(pdf_path: str, min_x_spread: int = 180, min_blocks: int = 8) -> bool:
    doc = fitz.open(str(pdf_path))

    try:
        for page in doc:
            blocks = page.get_text("blocks")
            text_blocks = []

            for block in blocks:
                x0, y0, x1, y1, text, block_no, block_type = block
                #x0: tọa độ trái, y0: tọa độ trên, x1: tọa độ phải, y1: tọa độ dưới của khối văn bản, text: nội dung text, block_no: số thứ tự khối, block_type: loại khối (0 là text, 1 là hình ảnh, 2 là vector)
                if block_type != 0:
                    continue

                if text and len(text.strip()) > 20:
                    text_blocks.append((x0, y0, x1, y1, text))

            if len(text_blocks) >= min_blocks:
                x_positions = [b[0] for b in text_blocks]
                x_spread = max(x_positions) - min(x_positions)

                if x_spread >= min_x_spread:
                    return True

        return False
    finally:
        doc.close()


#### 4.2 Mức 1 text-based PDF extractor

In [6]:
def extract_text_by_simple(pdf_path: Path) -> str:
    doc = fitz.open(pdf_path)
    pages_text = []

    for page in doc:
        text = page.get_text("text")
        if text:
            pages_text.append(text)

    doc.close()
    return normalize_pdf_text("\n".join(pages_text))


#### 4.2 Mức 2: scan PDF OCR word-level

Khi PDF là dạng scan thì ta không thể đọc trực tiếp, mà phải render ra ảnh sau đó đưa ảnh vào OCR

Hàm chuyển 1 trang pdf sang ảnh nhận đầu vào là 1 trang pdf và độ phân giải:
- 1 trang pdf mặc định có độ phân giải 72 DPI, nên để đạt được độ phân giải mong muốn, ta cần zoom lên theo tỷ lệ dpi/72
- Tạo ma trận hình học để phóng to pdf theo X và Y đúng tỉ lệ vừa tính (gấp dpi/72)
- Gói dữ liệu pixel thành 1 chuỗi byte theo chuẩn cấu trúc của png
- Chuyển đổi sang RGB để đảm bảo tesseract xử lý tốt hơn

In [7]:
def render_pdf_page_to_image(page: fitz.Page,
    dpi: int = OCR_RENDER_DPI, #dpi: độ phân giải
) -> Image.Image:
    zoom = dpi/72 
    matrix = fitz.Matrix(zoom, zoom) 
    pixmap = page.get_pixmap(matrix=matrix, alpha=False)
    image_bytes = pixmap.tobytes("png")
    return Image.open(io.BytesIO(image_bytes)).convert("RGB") 

show_image_for_debug: Dùng để hiển thị ảnh trong quá trinhg debug OCR
preprocess_image_for_ocr: Tiền xử lý ảnh trước khi đưa vào OCR
- Chuyển ảnh màu sang ảnh xám 
- Tự động tăng độ tương phản cho ảnh xám => Làm chữ đậm hơn, nền sáng hơn
- Lọc nhiễu bằng Median Filter => giúp loại bỏ các điểm nhiễu nhỏ
- Chuyển ảnh xác thành ảnh nhị phân đen trắng => Làm chữ rõ ràng hơn cho OCR
- Chuyển ảnh nhị phân về mode "1" về mode "L" (ảnh grayscale 8-bit, mode 1: ảnh 1-bit chỉ trắng đen)

In [32]:
def show_image_for_debug(
    image: Image.Image,
    title: str = "OCR Debug Image",
    figsize: tuple[int, int] = (10, 12),
) -> None:
    plt.figure(figsize=figsize)
    plt.imshow(image, cmap="gray")
    plt.title(title)
    plt.axis("off")
    plt.show()



Nhận một ảnh đầu vào -> xử lý ảnh -> chạy OCR -> lấy ra danh sách các từ đọc được kèm tọa độ và độ tin cậy
- Cho ảnh vào qua tiền xử lý trước
- Chạy OCR bằng Tesseract, trả về text kèm thông tin vị trí
- Duyệt qua từng từ trong ocr_data, lấy ra text và làm sạch text, lấy ra độ tin cậy => chuyển conf sang số
- Nếu độ tin cậy nhỏ hơn ngưỡng đưa ra thì bỏ qua từ đó
- Đồng thời Lấy tọa độ:
    - left: tọa độ x bên trái của từ
    - top: tọa độ y phía trên của từ
    - width: chiều rộng khung chứa từ
    - height: chiều cao khung chứa từ
- y_center = (y0 + y1)/2

In [10]:
# def extract_words_from_ocr_image(image: Image.Image, lang: str = OCR_LANG, psm: int = OCR_LAYOUT_PSM, min_confidence: int = OCR_MIN_CONFIDENCE, show_preprocessed: bool = False) -> list[dict[str, Any]]:
#     processed_image = preprocess_image_for_ocr(
#         image = image,
#         show_steps=show_preprocessed
#     )

#     ocr_data = pytesseract.image_to_data(
#         processed_image,
#         lang=lang,
#         config=f"--oem 3 --psm {psm}",
#         output_type=Output.DICT
#     )

#     valid_words: list[dict[str, Any]] = []
#     total_items = len(ocr_data["text"])

#     for index in range(total_items):
#         text = str(ocr_data["text"][index] or "").strip()
#         text = clean_line_text(text)

#         conf_raw = str(ocr_data["conf"][index]).strip()

#         try:
#             confidence = float(conf_raw)
#         except ValueError:
#             confidence = -1.0

#         if not text:
#             continue
#         if confidence < min_confidence:
#             continue

#         left = float(ocr_data["left"][index])
#         top = float(ocr_data["top"][index])
#         width = float(ocr_data["width"][index])
#         height = float(ocr_data["height"][index])

#         x0 = left
#         y0 = top
#         x1 = left + width
#         y1 = top + height
        
#         valid_words.append(
#             {
#                 "x0": x0,
#                 "y0": y0,
#                 "x1": x1,
#                 "y1": y1,
#                 "text": text,
#                 "width": width,  
#                 "height": height, 
#                 "y_center": (y0 + y1) / 2,
#                 "confidence": confidence,
#             }
#         )

#     return valid_words


import easyocr
import numpy as np
from PIL import Image
from typing import Any

# Khởi tạo Reader ở global scope để model chỉ load vào RAM 1 lần duy nhất.
# Thêm 'en' vì CV thường xuyên trộn lẫn cả tiếng Việt và tiếng Anh.
# Nếu máy bạn có GPU, hãy để gpu=True để tăng tốc độ nhận diện.
READER = easyocr.Reader(['vi', 'en'], gpu=False) 

def extract_words_with_easyocr(
    image: Image.Image, 
    min_confidence: float = 0.3 # Ngưỡng tự tin có thể để thấp hơn tesseract một chút
) -> list[dict[str, Any]]:
    
    # 1. EasyOCR làm việc tốt nhất với numpy array (định dạng của OpenCV)
    # Ta không cần dùng hàm preprocess_image_for_ocr làm mờ/nhị phân hóa nữa, 
    # cứ đưa thẳng ảnh gốc vào để giữ nguyên vẹn dấu tiếng Việt.
    img_np = np.array(image)
    
    # 2. Thực thi OCR
    # detail=1 trả về Bounding Box, Text, và Confidence
    results = READER.readtext(img_np, detail=1)
    
    valid_words: list[dict[str, Any]] = []
    
    for bbox, text, conf in results:
        # Làm sạch text
        text = str(text).strip()
        confidence = float(conf)
        
        if not text:
            continue
        if confidence < min_confidence:
            continue
            
        # 3. Xử lý Bounding Box
        # EasyOCR trả về bbox dạng 4 điểm: [top_left, top_right, bottom_right, bottom_left]
        # Mỗi điểm là một list/tuple [x, y]
        top_left = bbox[0]
        bottom_right = bbox[2]
        
        x0 = float(top_left[0])
        y0 = float(top_left[1])
        x1 = float(bottom_right[0])
        y1 = float(bottom_right[1])
        
        width = x1 - x0
        height = y1 - y0
        
        valid_words.append(
            {
                "x0": x0,
                "y0": y0,
                "x1": x1,
                "y1": y1,
                "text": text,
                "width": width,  
                "height": height, 
                "y_center": (y0 + y1) / 2,
                "confidence": confidence,
            }
        )

    return valid_words

Using CPU. Note: This module is much faster with a GPU.


Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

group_words_into_rows: Nhóm các từ thành 1 dòng
input: danh sách từ rời rạc
output: Danh sách gồm nhiều dòng (dòng nó lại 1 danh sách các từ chứ không phải câu)
- sắp xếp các từ theo: y_center tăng dần (từ trên xuống dưới) -> x0 tăng dần (trong cùng vị trí dọc thì từ trái sang phải)
- Duyệt qua từng từ trong words:
    - Nếu chưa có từ nào trong current_rows => Thêm từ hiện tại vào làm từ đầu và bỏ qua tới vòng lặp tiếp theo
    - Lấy y_center của các từ trong current_rows ra, lấy y_center min và max
    - Sau đó kiểm tra y_center của từ hiện tại có nằm trong ngưỡng không => Có thì thêm vào current_rows

In [13]:
def group_words_into_rows(
    words: list[dict[str, Any]],
    y_tolerance: float = 20.0,
) -> list[list[dict[str, Any]]]:
    words = sorted(words, key=lambda w: (w["y_center"], w["x0"]))

    rows = []
    current_row = []

    for word in words:
        if not current_row:
            current_row = [word]
            continue

        row_y_values = [w["y_center"] for w in current_row]
        row_min_y = min(row_y_values)
        row_max_y = max(row_y_values)

        # Cho phép từ mới nằm trong dải Y mở rộng của row hiện tại
        if row_min_y - y_tolerance <= word["y_center"] <= row_max_y + y_tolerance:
            current_row.append(word)
        else:
            rows.append(sorted(current_row, key=lambda w: w["x0"]))
            current_row = [word]

    if current_row:
        rows.append(sorted(current_row, key=lambda w: w["x0"]))

    return rows


rows_to_lines_by_x_gap: Thông thường khoảng cách giữa 2 từ trên 1 dòng có thể cách nhau 1 khoảng lớn (bố cục 2 cột). Nếu vậy chúng ta sẽ chia làm 2 dòng riêng
- Chúng ta sẽ lấy các rows đã thu được ở group_words_into_rows
- Lấy ra từng row trong rows, duyệt qua từng từ. Sau đó kiểm tra khoangr cách x1 của từ trước và x0 của từ sau. Nếu vượt ngưỡng x_pre thì sẽ bị tách 2 dòng. Còn nếu không thì chúng ở trên 1 dòng. 
- Lấy danh sách các từ trên 1 dòng cho qua make_line_from_words để nó gôpj thành 1 dòng câu luôn
=> Output: Nó sẽ lấy ra danh sách các text

make_line_from_words: Nó sẽ nhận các danh sách các từ => gom lại thành 1 dòng câu luôn
- Sắp xếp tập words tăng dần theo x0 => Gom các word trong words thành 1 text, rồi làm sạch nó.
=> Trả về tọa độ của cả text đó, x0: tọa độ trái nhất, y0: tọa độ trên trái nhất, x1: tọa độ phải nhất, y1: tạ độ dưới nhất. Và cả text

rows_to_column_lines: Tách thành 2 cột, trái/phải (column_margin: khoảng cách lề, threshold = right_start_x - column+margin: tức là tọa độ x của từ)
- left_line: Các dòng bên cột trái
- right_line: Các dòng bên cột phải



In [14]:
# Gộp một danh sách các từ thành một dòng duy nhất, tính toán bounding box tổng thể và nối các từ lại với nhau để tạo thành văn bản của dòng đó
def make_line_from_words(words: list[dict[str, Any]]) -> Optional[dict[str, Any]]:
    if not words:
        return None

    words = sorted(words, key=lambda w: w["x0"])
    text = " ".join(w["text"] for w in words)
    text = clean_line_text(text)

    if not text:
        return None

    return {
        "x0": min(w["x0"] for w in words),
        "y0": min(w["y0"] for w in words),
        "x1": max(w["x1"] for w in words),
        "y1": max(w["y1"] for w in words),
        "text": text,
    }

# Gộp các từ trong mỗi dòng thành một dòng duy nhất, nhưng chỉ tách chúng thành các dòng riêng biệt nếu khoảng cách x giữa chúng vượt quá một ngưỡng nhất định (x_gap_threshold)    
def rows_to_lines_by_x_gap(rows: list[list[dict[str, Any]]], x_gap_threshold: float = 50.0) -> list[dict[str, Any]]:
    lines = []

    for row in rows:
        current_segment = []
        prev_x1 = None

        for word in row:
            if prev_x1 is not None and word["x0"] - prev_x1 > x_gap_threshold:
                line = make_line_from_words(current_segment)
                if line:
                    lines.append(line)
                current_segment = [word]
            else:
                current_segment.append(word)

            prev_x1 = word["x1"]

        if current_segment:
            line = make_line_from_words(current_segment)
            if line:
                lines.append(line)

    return lines

# Dựa trên vị trí bắt đầu của cột phải (right_start_x) và một khoảng cách lề (column_margin), phân loại các từ trong mỗi dòng thành cột trái hoặc cột phải, sau đó gộp chúng thành các dòng hoàn chỉnh cho từng cột. Trả về hai danh sách các dòng cho cột trái và cột phải, được sắp xếp theo thứ tự xuất hiện trên trang.
def rows_to_column_lines(rows: list[list[dict[str, Any]]], right_start_x: float, column_margin: float = 8.0) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    left_lines = []
    right_lines = []
    threshold = right_start_x - column_margin

    for row in rows:
        left_words = []
        right_words = []

        for word in row:
            if word["x0"] >= threshold:
                right_words.append(word)
            else:
                left_words.append(word)

        left_line = make_line_from_words(left_words)
        right_line = make_line_from_words(right_words)

        if left_line:
            left_lines.append(left_line)
        if right_line:
            right_lines.append(right_line)

    left_lines.sort(key=lambda line: (line["y0"], line["x0"]))
    right_lines.sort(key=lambda line: (line["y0"], line["x0"]))
    return left_lines, right_lines



detect_two_column_layout: Phát hiện layout 2 cột
- rough_lines: danh sách các dòng đã được gom ( {"x0": 40, "y0": 100, "x1": 250, "y1": 115, "text": "Experience..."},)
- Lấy x0 của các dòng:
    - Làm tròn x0
    - Đếm tần suất từng cụm x
    - Chọn các x xuất hiện đủ nhiều
    - Tìm khoảng cách lớn nhất giữa các x ứng viên
    - Nếu khoảng cách đủ lớn thì bị xem là 2 cột

In [15]:
#Phân tích tọa độ X để đoán xem trang này có 2 cột không. Nếu có, trả về vị trí X của cột trái và cột phải, cũng như điểm giữa để tách cột. Nếu không chắc chắn, trả về None.
def detect_two_column_layout(rough_lines: list[dict[str, Any]], page_width: float) -> Optional[dict[str, float]]:
    x_bins = []

    for line in rough_lines:
        text = line["text"]

        if len(text) < 2:
            continue

        x_bin = round(line["x0"] / 5) * 5
        x_bins.append(x_bin)

    if len(x_bins) < 2:
        return None

    counter = Counter(x_bins)
    min_count = max(2, int(len(rough_lines) * 0.04))

    candidates = sorted(x for x, count in counter.items() if count >= min_count)
    if len(candidates) < 2:
        return None

    gaps = []
    for left_x, right_x in zip(candidates, candidates[1:]):
        gap = right_x - left_x

        if left_x > page_width * 0.35 and right_x > page_width * 0.78:
            continue

        gaps.append((gap, left_x, right_x))

    if not gaps:
        return None

    max_gap, left_start, right_start = max(gaps, key=lambda item: item[0])
    min_gap = max(55, page_width * 0.08)

    if max_gap < min_gap:
        return None

    return {
        "left_start": float(left_start),
        "right_start": float(right_start),
        "split_x": float((left_start + right_start) / 2),
    }

looks_line_new_logical_line: Đoán xem 1 dòng text có phải là 1 điểm bắt đầu của 1 dòng logic mới hay không

compute_line_stats: Hàm này tính một số thống kê cơ bản của các dòng
- Tính chiều cau của từng dòng => bounding box: y0 và y1 của dòng đó => chiều cao nó sẽ tính là y1 - y0
- Tính khoảng cách dọc giữa các dòng và thêm vào gaps
- def median: Hàm tính trung vị
    - Nếu như số lượng giá trị trong danh sách là số lẻ => trung vị nó sẽ là giá trị ở giữa
    - Ngược lại, nó sẽ return float(values[mid - 1] + values[mid]) / 2
=> Dòng trung bình cao khoảng bao nhiêu và Khoảng cách giữa các dòng thường khoảng bao nhiêu 

is_wrapped_continuation: quyết định next_line có phải phần nối tiếp của current_line không
- Lấy thống kê dòng trung bình cao cao nhiêu, khoảng cách giữa các dòng
- Tính khoảng cách dọc và độ lệch X:
    - vertical_gap: Khoảng cách giữa dòng trước và dòng tiếp theo
    - x_shift: Độ lệch dòng đầu vào của dòng trước và dòng tiếp theo
- Tạo ngưỡng khoảng cách nhỏ, quyết định 2 dòng có đủ gần nhau không (small_gap_threshold)
- Tính độ lệch cột x
- Nếu vertial_gap vượt ra khỏi small_gap.. hoặc x_shift vượt khỏi độ lệch cột thì trả về false

merge_wrapped_lines: Gộp các dòng bị xuống hàng 
- Sắp xếp dòng theo thứ tự đọc, từ trên xuống dưới, từ trái qua phải
- Tính thống kê stas: Chiều cao, khoảng cách giữa các dòng
- Tạo list kết quả và lấy dòng đầu tiên làm current
- Duyệt từng dòng tiếp theo:
    - Kiểm tra next_line có là phần tiếp không => Nếu có thì thực hiện ghép 2 dòng lại
    

In [64]:
def looks_like_new_logical_line(text: str) -> bool:
    if not text:
        return False

    stripped = text.strip()
    normalized = normalize_for_heading(stripped)

    if not normalized:
        return False

    if detect_section_heading(stripped):
        return True

    if re.match(r"^[-*•]", stripped):
        return True

    if YEAR_RANGE_RE.search(normalized):
        return True

    if re.match(
        r"^(?:[A-Z][A-Za-z0-9&/()\\-]+\\s+){1,5}[A-Z][A-Za-z0-9&/()\\-]+$",
        stripped,
    ):
        return True
    

    return False
def compute_line_stats(lines: list[dict[str, Any]]) -> dict[str, float]:
    if not lines:
        return {
            "median_height": 0.0,
            "median_gap": 0.0,
        }

    heights = [max(1.0, line["y1"] - line["y0"]) for line in lines]

    gaps = []
    for prev_line, next_line in zip(lines, lines[1:]):
        gap = next_line["y0"] - prev_line["y1"]
        if gap >= 0:
            gaps.append(gap)

    def median(values: list[float]) -> float:
        if not values:
            return 0.0
        values = sorted(values)
        mid = len(values) // 2
        if len(values) % 2 == 1:
            return float(values[mid])
        return float(values[mid - 1] + values[mid]) / 2

    return {
        "median_height": median(heights),
        "median_gap": median(gaps),
    }

def is_wrapped_continuation(
    current_line: dict[str, Any],
    next_line: dict[str, Any],
    stats: dict[str, float],
) -> bool:
    current_text = current_line["text"].strip()
    next_text = next_line["text"].strip()

    if not current_text or not next_text:
        return False

    if detect_section_heading(next_text):
        return False

    # Neu dong sau trong giong title / moc thoi gian moi thi khong merge
    if YEAR_RANGE_RE.search(normalize_for_heading(next_text)):
        return False
    if looks_like_new_logical_line(next_text):
        return False

    # Neu dong sau bat dau bang bullet manh thi thuong la y moi
    if re.match(r"^[-*•]", next_text):
        return False

    median_height = max(1.0, stats.get("median_height", 0.0))
    median_gap = stats.get("median_gap", 0.0)

    vertical_gap = max(0.0, next_line["y0"] - current_line["y1"])
    x_shift = abs(next_line["x0"] - current_line["x0"])

    # Nguong gap nho -> nghieng ve kha nang la xuong dong tu dong
    small_gap_threshold = max(6.0, median_gap * 1.5, median_height * 0.35)

    # Lech cot qua nhieu thi khong merge
    max_same_column_shift = max(18.0, median_height * 1.2)

    if vertical_gap > small_gap_threshold:
        return False

    if x_shift > max_same_column_shift:
        return False
    
    # Neu dong truoc ket thuc ro rang thi thuong la y da xong
    if re.search(r"[.!?:;]$", current_text):
        return False
    
    if re.search(r"[,]$", current_text):
        return True

    # if vertical_gap <= small_gap_threshold or x_shift <= max_same_column_shift:
    # # Neu dong sau mo dau bang chu thuong / so / ky tu tiep noi
    #     if re.match(r"^[a-z0-9(,/+-]", next_text):
    #         return True

    return False

def merge_wrapped_lines(lines: list[dict[str, Any]]) -> list[dict[str, Any]]:
    if not lines:
        return []

    sorted_lines = sorted(lines, key=lambda line: (line["y0"], line["x0"]))
    stats = compute_line_stats(sorted_lines)

    merged_lines: list[dict[str, Any]] = []
    current = dict(sorted_lines[0])

    for next_line in sorted_lines[1:]:
        if is_wrapped_continuation(current, next_line, stats):
            current["text"] = clean_line_text(current["text"] + " " + next_line["text"])
            current["x0"] = min(current["x0"], next_line["x0"])
            current["y0"] = min(current["y0"], next_line["y0"])
            current["x1"] = max(current["x1"], next_line["x1"])
            current["y1"] = max(current["y1"], next_line["y1"])
        else:
            merged_lines.append(current)
            current = dict(next_line)

    merged_lines.append(current)
    return merged_lines


#### 4.3 Mức 3: layout-aware extractor dùng chung

detect_section_heading: Kiểm tra dòng text có phải tiêu đề không

parse_section_from_line: 
- Input: Danh sách các text đã xử lý
- Output: 
    - leading_lines: Các dòng nằm trước section đầu tiên
    - sections: nội dung đã chia theo section

merge_sections: Gộp section từ source vào target (Gộp section cột trái và phải)


In [17]:
def detect_section_heading(text: str) -> Optional[str]:
    norm = normalize_for_heading(text)

    if not norm:
        return None

    for section, aliases in NORMALIZED_SECTION_ALIASES.items():
        if norm in aliases:
            return section

    return None

# Phân tích các dòng đã gộp để phát hiện các tiêu đề phần dựa trên văn bản của chúng, sau đó phân loại các dòng thành phần dẫn đầu (trước tiêu đề phần đầu tiên) và các phần tương ứng dựa trên tiêu đề đã phát hiện. Trả về một danh sách các dòng dẫn đầu và một dictionary ánh xạ tên phần sang danh sách các dòng thuộc phần đó.
def parse_sections_from_lines(lines: list[dict[str, Any]]) -> tuple[list[str], dict[str, list[str]]]:
    leading_lines = []
    sections = {}
    current_section = None

    for line in lines:
        text = line["text"].strip()

        if not text:
            continue

        detected_section = detect_section_heading(text)
        if detected_section:
            current_section = detected_section
            sections.setdefault(current_section, [])
            continue

        if current_section is None:
            leading_lines.append(text)
        else:
            sections.setdefault(current_section, []).append(text)

    return leading_lines, sections

# Hợp nhất các phần từ một nguồn vào phần tương ứng trong đích, thêm các dòng vào danh sách hiện có cho phần đó. Nếu phần chưa tồn tại trong đích, tạo nó trước khi thêm các dòng.
def merge_sections(target: dict[str, list[str]], source: dict[str, list[str]]) -> None:
    for section, lines in source.items():
        target.setdefault(section, [])
        target[section].extend(lines)

# Loại bỏ các dòng trùng lặp trong một danh sách, nhưng giữ nguyên thứ tự xuất hiện của chúng. Trả về một danh sách mới chỉ chứa các dòng duy nhất theo thứ tự ban đầu.
def remove_duplicate_keep_order(lines: list[str]) -> list[str]:
    result = []
    seen = set()

    for line in lines:
        key = normalize_for_heading(line)

        if not key:
            continue
        if key in seen:
            continue

        seen.add(key)
        result.append(line)

    return result

# Xây dựng văn bản thuần túy cuối cùng từ các phần đã trích xuất, sắp xếp chúng theo thứ tự ưu tiên đã định trước, loại bỏ các dòng trùng lặp trong mỗi phần và thêm tiêu đề phần tương ứng trước mỗi phần. Kết quả là một chuỗi văn bản đã được làm sạch và tổ chức tốt.
def build_plain_text(sections: dict[str, list[str]]) -> str:
    parts = []

    for section in PREFERRED_SECTION_ORDER:
        lines = sections.get(section, [])
        lines = remove_duplicate_keep_order(lines)

        if not lines:
            continue

        title = SECTION_TITLES.get(section, section.upper())
        parts.append(title)
        parts.extend(lines)
        parts.append("")

    return clean_cv_text("\n".join(parts))


append_layout_result_from_words:
- Gom các từ thành danh sách các row => rows
- Tạo rough_lines, Nó sẽ thực hiện lấy rows => Nó sẽ thực hiện kiểm tra xem có bị tách thành 2 cột không => tạo 1 danh sách rough lines mới chứa các row
- Thực hiện đoán layout 2 cột, trả về tọa độ bắt đầu của cột trái, cộtphải, khoảng cách giữa 2 cột
- Nếu phát hiện layout 2 cột:
    - Chia rows thành cột trái và cột phải bằng rows_to_column_lines với đầu vào là list rows => Trả về danh sách rows của cột trái và cột phải
    - Merge các dòng bị wrap trong từng cột
    - Parse section từng cột => trả về nội dung tương ứng với section, leadinh
    - Đưa leading vào section header, các sextions khác thì đưa vào sexecsion orther
    - Gộp section của 2 cột vào result
- Ngược lại nếu không phát hiện 2 cột:
    - Xử lý như một cột: lines = row_to_single_column_lines      


In [18]:
# Gộp các từ trong mỗi dòng thành một dòng duy nhất, không tách chúng thành các dòng riêng biệt dựa trên khoảng cách x, sau đó sắp xếp tất cả các dòng theo thứ tự xuất hiện trên trang (đầu tiên theo y0, sau đó theo x0).
def rows_to_single_column_lines(rows: list[list[dict[str, Any]]]) -> list[dict[str, Any]]:
    lines = []

    for row in rows:
        line = make_line_from_words(row)
        if line:
            lines.append(line)

    lines.sort(key=lambda line: (line["y0"], line["x0"]))
    return lines


def append_layout_result_from_words(
    result: dict[str, Any],
    words: list[dict[str, Any]],
    page_width: float,
    page_index: int,
    y_tolerance: float = 20.0,
    x_gap_threshold: float = 50.0,
    column_margin: float = 8.0,
) -> None:
    if not words:
        result["warnings"].append(f"Trang {page_index + 1} không lấy được word nào.")
        return

    rows = group_words_into_rows(words, y_tolerance=y_tolerance)
    rough_lines = rows_to_lines_by_x_gap(rows, x_gap_threshold=x_gap_threshold)
    layout = detect_two_column_layout(rough_lines, page_width=page_width)

    if layout:
        left_lines, right_lines = rows_to_column_lines(
            rows,
            right_start_x=layout["right_start"],
            column_margin=column_margin,
        )

        left_lines = merge_wrapped_lines(left_lines)
        right_lines = merge_wrapped_lines(right_lines)

        left_leading, left_sections = parse_sections_from_lines(left_lines)
        right_leading, right_sections = parse_sections_from_lines(right_lines)

        if page_index == 0:
            result["sections"]["header"].extend(left_leading + right_leading)
        else:
            result["sections"]["other"].extend(left_leading + right_leading)

        merge_sections(result["sections"], left_sections)
        merge_sections(result["sections"], right_sections)

        result["layout_debug"].append(
            {
                "page": page_index + 1,
                "layout": "two_columns_from_ocr_words",
                "left_start": layout["left_start"],
                "right_start": layout["right_start"],
                "split_x": layout["split_x"],
                "left_line_count": len(left_lines),
                "right_line_count": len(right_lines),
                "word_count": len(words),
            }
        )
        return

    lines = rows_to_single_column_lines(rows)
    lines = merge_wrapped_lines(lines)
    leading_lines, sections = parse_sections_from_lines(lines)

    if page_index == 0:
        result["sections"]["header"].extend(leading_lines)
    else:
        result["sections"]["other"].extend(leading_lines)

    merge_sections(result["sections"], sections)

    result["layout_debug"].append(
        {
            "page": page_index + 1,
            "layout": "single_column_from_ocr_words",
            "line_count": len(lines),
            "word_count": len(words),
        }
    )
    


get_valid_words: Lấy toàn bộ các từ trên trang PDF với tọa độ bouding box

extract_cv_layout_aware: Hàm chính để trích xuất văn bản từ cv nhiều bố cục
- Nó nhận đường dẫn PDF, đọc từng trang, xử lý layout, rồi trả về dictionary kết quả.
- Kiểm tra pdf có phải scan không. Nếu không:
    - lặp qua từng trang
    - Lấy word hợp lệ từng trang, gom word thành row => ĐƯợc 1 danh sách gồm các rows
    - Tạo rough_line để kiểm tra detect
    - Nếu 2 cột thì mình thực hiện như cũ 

In [19]:
# Lấy toàn bộ các từ trên một trang PDF cùng với tọa độ bounding box của chúng (x0, y0, x1, y1)
def get_valid_words(page) -> list[dict[str, Any]]:
    raw_words = page.get_text("words")
    valid_words = []

    for w in raw_words:
        x0, y0, x1, y1, word, *_ = w
        word = clean_line_text(str(word))

        if not word:
            continue

        valid_words.append({
            "x0": float(x0),
            "y0": float(y0),
            "x1": float(x1),
            "y1": float(y1),
            "text": word,
            "y_center": (float(y0) + float(y1)) / 2,
        })

    return valid_words

# Hàm chính để trích xuất văn bản từ CV PDF, có khả năng nhận biết bố cục nhiều cột và các phần khác nhau. Nó sử dụng PyMuPDF để đọc PDF, phân tích bố cục dựa trên tọa độ của các từ, và tổ chức văn bản thành các phần có ý nghĩa như thông tin cá nhân, kỹ năng, kinh nghiệm, v.v. Kết quả là một dictionary chứa văn bản thuần túy đã được tổ chức theo phần, cùng với thông tin về loại PDF và bất kỳ cảnh báo nào nếu có.
def extract_cv_layout_aware(pdf_path: str | Path, y_tolerance: float = 4.0, x_gap_threshold: float = 50.0, column_margin: float = 8.0) -> dict[str, Any]:
    pdf_path = str(pdf_path)

    if not Path(pdf_path).exists():
        raise FileNotFoundError(f"File not found: {pdf_path}")

    result = {
        "text": "",
        "sections": {section: [] for section in PREFERRED_SECTION_ORDER},
        "pdf_type": "multi_column_or_complex_layout",
        "method": "pymupdf_words_layout_level3",
        "warnings": [],
        "layout_debug": [],
    }

    if is_probably_scanned_pdf(pdf_path):
        result["pdf_type"] = "scan_or_image_based"
        result["method"] = "none"
        result["warnings"].append("PDF co ve la file scan/anh hoac co qua it text. Can OCR de doc noi dung.")
        return result

    doc = fitz.open(pdf_path)

    try:
        for page_index, page in enumerate(doc):
            words = get_valid_words(page)

            if not words:
                result["warnings"].append(f"Trang {page_index + 1} khong lay duoc word nao.")
                continue

            rows = group_words_into_rows(words, y_tolerance=y_tolerance)
            rough_lines = rows_to_lines_by_x_gap(rows, x_gap_threshold=x_gap_threshold)
            layout = detect_two_column_layout(rough_lines, page_width=page.rect.width)

            if layout:
                left_lines, right_lines = rows_to_column_lines(rows, right_start_x=layout["right_start"], column_margin=column_margin)

                left_leading, left_sections = parse_sections_from_lines(left_lines)
                right_leading, right_sections = parse_sections_from_lines(right_lines)

                if page_index == 0:
                    result["sections"]["header"].extend(left_leading)
                    result["sections"]["header"].extend(right_leading)
                else:
                    result["sections"]["other"].extend(left_leading)
                    result["sections"]["other"].extend(right_leading)

                merge_sections(result["sections"], left_sections)
                merge_sections(result["sections"], right_sections)

                result["layout_debug"].append({
                    "page": page_index + 1,
                    "layout": "two_columns",
                    "left_start": layout["left_start"],
                    "right_start": layout["right_start"],
                    "split_x": layout["split_x"],
                    "left_line_count": len(left_lines),
                    "right_line_count": len(right_lines),
                })
            else:
                lines = rows_to_single_column_lines(rows)
                lines = merge_wrapped_lines(lines)
                leading, sections = parse_sections_from_lines(lines)

                if page_index == 0:
                    result["sections"]["header"].extend(leading)
                else:
                    result["sections"]["other"].extend(leading)

                merge_sections(result["sections"], sections)

                result["layout_debug"].append({
                    "page": page_index + 1,
                    "layout": "single_column",
                    "line_count": len(lines),
                })
    finally:
        doc.close()

    for section, lines in result["sections"].items():
        result["sections"][section] = remove_duplicate_keep_order(lines)

    result["text"] = build_plain_text(result["sections"])

    if len(result["text"]) < 300:
        result["warnings"].append("Text sau trich xuat kha ngan. Co the PDF bi scan, bi khoa hoac layout qua phuc tap.")

    return result


extract_text_from__scanned_pdf_layout_aware: 
- PDF page → render thành ảnh → OCR ảnh → lấy words + tọa độ → xử lý layout
- Thực hiện xây dựng 1 khung kết quả gồm các text, sections
- Mở fiele pdf
- Duyệt qua từng trang
    - Render trang pdf thành ảnh
    - Cho ảnh qua OCR tar về danh sách word có tọa độ
    - Thực hiện gom các từ:
        → group_words_into_rows
        → rows_to_lines_by_x_gap
        → detect_two_column_layout
        → nếu 2 cột: rows_to_column_lines
        → nếu 1 cột: rows_to_single_column_lines
        → merge_wrapped_lines
        → parse_sections_from_lines
        → merge_sections vào result
        → ghi layout_debug

In [25]:
def extract_text_from_scanned_pdf_layout_aware(
    pdf_path: str | Path,
    dpi: int = OCR_RENDER_DPI,
    lang: str = OCR_LANG,
    psm: int = OCR_LAYOUT_PSM,
    min_confidence: int = OCR_MIN_CONFIDENCE,
    show_preprocessed_first_page: bool = False,
    y_tolerance: float = 20.0,
    x_gap_threshold: float = 50.0,
    column_margin: float = 8.0,
) -> dict[str, Any]:
    pdf_path = Path(pdf_path)

    if not pdf_path.exists():
        raise FileNotFoundError(f"Không tìm thấy file: {pdf_path}")

    result = build_extraction_result(
        pdf_type="scan_or_image_based",
        method=f"ocr_words_layout_psm_{psm}",
        warnings=["Đã dùng OCR word-level vì PDF có vẻ là file scan/ảnh."],
    )

    document = fitz.open(str(pdf_path))

    try:
        for page_index, page in enumerate(document):
            image = render_pdf_page_to_image(page, dpi=dpi)
            words = extract_words_with_easyocr(
                image=image,
                min_confidence=min_confidence,
            )   

            append_layout_result_from_words(
                result=result,
                words=words,
                page_width=image.width,
                page_index=page_index,
                y_tolerance=y_tolerance,
                x_gap_threshold=x_gap_threshold,
                column_margin=column_margin,
            )
    finally:
        document.close()

    for section, lines in result["sections"].items():
        result["sections"][section] = dedupe_lines_keep_order(lines)

    result["text"] = build_plain_text(result["sections"])

    if len(result["text"]) < OCR_MIN_TEXT_LENGTH:
        result["warnings"].append(
            "Text OCR khá ngắn. Có thể ảnh mờ, lệch, hoặc layout scan quá khó."
        )

    return result


#### 4.4 Orchestrator chọn nhánh xử lý

In [21]:
def extract_text_from_pdf(pdf_path: Path | str) -> dict[str, Any]:
    pdf_path = str(pdf_path)

    if not Path(pdf_path).exists():
        raise FileNotFoundError(f"File not found: {pdf_path}")

    if is_probably_scanned_pdf(pdf_path):
        return extract_text_from_scanned_pdf_layout_aware(pdf_path, show_preprocessed_first_page=True)

    if is_probably_multi_column(pdf_path):
        return extract_cv_layout_aware(pdf_path)

    text = extract_text_by_simple(pdf_path)
    result = {
        "text": text,
        "sections": {section: [] for section in PREFERRED_SECTION_ORDER},
        "pdf_type": "text_based",
        "method": "pymupdf_text",
        "warnings": [],
        "layout_debug": [],
    }

    if len(text) < 300:
        result["warnings"].append("Extracted text is short. Check whether the PDF is scanned, protected, or has complex layout.")

    return result


In [65]:
pdf_path = r"C://Users//Admin//Downloads//CV_NTTung.pdf"

extraction_result = extract_text_from_pdf(pdf_path)

print("PDF type:", extraction_result["pdf_type"])
print("Method:", extraction_result["method"])
print("Warnings:", extraction_result["warnings"])
    
# Save all extracted text to a txt file
output_path = Path(r"C://Users//Admin//Downloads//CV_NTTung_extracted.txt")

with open(output_path, "w", encoding="utf-8") as f:
    f.write("PDF type: " + str(extraction_result["pdf_type"]) + "\n")
    f.write("Method: " + str(extraction_result["method"]) + "\n")
    f.write("Warnings: " + str(extraction_result["warnings"]) + "\n")
    f.write("-" * 80 + "\n\n")
    f.write(extraction_result["text"])

print("Da luu toan bo text vao file:")
print(output_path)


PDF type: scan_or_image_based
Method: ocr_words_layout_psm_4
Warnings: ['Đã dùng OCR word-level vì PDF có vẻ là file scan/ảnh.']
Da luu toan bo text vao file:
C:\Users\Admin\Downloads\CV_NTTung_extracted.txt


### 5. Post-extraction cleaning

is_noise_line: Kiểm tra xem 1 dòng có phải dòng rác không

clean_section__lines: Hàm nhận vào danh sách các dòng text và trả về danh sách đã làm sạch

In [27]:
def is_noise_line(text: str) -> bool:
    if not text:
        return True

    normalized = normalize_for_heading(text)
    if not normalized or len(normalized) <= 1:
        return True

    return bool(re.fullmatch(r"[-_ ]+", text))

def clean_section_lines(lines: list[str]) -> list[str]:
    cleaned: list[str] = []

    for line in lines:
        line = clean_line_text(line)
        if is_noise_line(line):
            continue
        cleaned.append(line)

    return dedupe_lines_keep_order(cleaned)


#### Làm sạch toàn bộ extraction result

In [28]:
def clean_extraction_result(extraction_result: dict[str, Any]) -> dict[str, Any]:
    cleaned_result = {
        "text": "",
        "sections": {},
        "pdf_type": extraction_result.get("pdf_type", ""),
        "method": extraction_result.get("method", ""),
        "warnings": list(extraction_result.get("warnings", [])),
        "layout_debug": extraction_result.get("layout_debug", []),
    }

    raw_sections = extraction_result.get("sections", {})
    if not raw_sections:
        raw_sections = {section: [] for section in PREFERRED_SECTION_ORDER}

    cleaned_sections = {}

    for section in PREFERRED_SECTION_ORDER:
        lines = raw_sections.get(section, [])
        cleaned_sections[section] = clean_section_lines(lines)

    cleaned_result["sections"] = cleaned_sections
    cleaned_result["text"] = build_plain_text(cleaned_sections)

    if len(cleaned_result["text"]) < 300:
        cleaned_result["warnings"].append(
            "Cleaned text is still short. Check extraction quality before moving to field extraction."
        )

    return cleaned_result


In [31]:
pdf_path = r"C:\Users\Admin\Downloads\CV_NTTung.pdf"

extraction_result = extract_text_from_pdf(pdf_path)
cleaned_result = clean_extraction_result(extraction_result)

print("PDF type:", cleaned_result["pdf_type"])
print("Method:", cleaned_result["method"])
print("Warnings:", cleaned_result["warnings"])
print("-" * 80)
print(cleaned_result["text"][:5000])

output_path = Path(r"C:\Users\Admin\Downloads\CV_NTTung_cleaned.txt")

with open(output_path, "w", encoding="utf-8") as f:
    f.write("PDF type: " + str(cleaned_result["pdf_type"]) + "\n")
    f.write("Method: " + str(cleaned_result["method"]) + "\n")
    f.write("Warnings: " + str(cleaned_result["warnings"]) + "\n")
    f.write("-" * 80 + "\n\n")
    f.write(cleaned_result["text"])

print("Da luu cleaned text vao file:")
print(output_path)

e:\Semester6\Laptrinhpython\project\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


PDF type: scan_or_image_based
Method: ocr_words_layout_psm_4
Warnings: ['Đã dùng OCR word-level vì PDF có vẻ là file scan/ảnh.']
--------------------------------------------------------------------------------
THONG TIN DAU CV
PHÁP CHẾ
Có thể ứng dụng các kỹ năng, kinh nghiệm trong linh vực luật, phảp lý để trở thành luật sư chuyên nghiệp.
Mong muõn làm việc lâu dài, trở thành chuyên gia trong lĩnh vực pháp lý đảm nhận.
NGUYỄN THANH
TÙNG
& Nam
13/01/1995 0324 879 412 lanhanh95 @gmail .com
Vĩnh Yên; Vĩnh Phúc
TRÌNH ĐỘ HỌC VẤN
Đại học Luật Hà Nội.
( 2013-2017 )
Chuyên ngành: Luật kinh tế.
Tõt nghiệp loại giỏí.

KINH NGHIEM LAM VIEC
Công ty Trách nhiệm hữu hạn TT.
12/2018 nay.
Vị trí: Thủ tục pháp lý bất động sản.
Văn phòng Công chứng Đại Hưng .
12/2017 07/2018
Vị trí: Thực tập sinh.

KY NANG
Lảm việc nhóm.
Kỹ nẳng thuyểt trinh.
Kỹ năng quản lỷ dự án.

CHUNG CHI
Chúng chỉ tiếng Anh chuyên ngành.
Chứng chỉ Ứng dụng Công nghệ thông tin.
GIẢl THƯỞNG
Đạt giải nhẩt trong cuộc thi Sinh viên ngh

### 6. Tải tài nguyên spaCy

load_translated_dictionary: nhận vào đường dẫn và trả về 1 dictionary

In [35]:
import csv
import spacy
from spacy.matcher import PhraseMatcher
from pathlib import Path
from typing import Any

nlp = spacy.load("en_core_web_sm")

DATA_DIR = Path(r"E:\Semester6\Laptrinhpython\project\data")

def load_translated_dictionary(csv_path: Path) -> list[dict[str, str]]:
    rows = []

    with csv_path.open("r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)

        for row in reader:
            rows.append({
                "english": str(row.get("english", "") or "").strip(),
                "vietnamese": str(row.get("vietnamese", "") or "").strip(),
            })

    return rows

title_entries = load_translated_dictionary(DATA_DIR / "title_dict_translated_vi.csv")
skill_entries = load_translated_dictionary(DATA_DIR / "skill_dict_translated_vi.csv")

print("title_entries:", len(title_entries))
print("skill_entries:", len(skill_entries))
print(title_entries[:5])
print(skill_entries[:5])


title_entries: 896
skill_entries: 640
[{'english': '3d printing technician', 'vietnamese': 'kỹ thuật viên in 3d'}, {'english': 'accountant', 'vietnamese': 'kế toán viên'}, {'english': 'accounts receivable specialist', 'vietnamese': 'chuyên viên kế toán phải thu'}, {'english': 'actor', 'vietnamese': 'diễn viên'}, {'english': 'administrative assistant', 'vietnamese': 'trợ lý hành chính'}]
[{'english': '3d printing', 'vietnamese': 'in 3d'}, {'english': 'a/b testing', 'vietnamese': 'thử nghiệm a/b'}, {'english': 'accessibility', 'vietnamese': 'khả năng tiếp cận'}, {'english': 'accessibility standards', 'vietnamese': 'tiêu chuẩn tiếp cận'}, {'english': 'accessibility testing', 'vietnamese': 'kiểm tra khả năng tiếp cận'}]


vuild_bilingual_lookup: Hàm nahanj vào danh sách dữ liệu dạng
[
    {"english": "Software Engineer", "vietnamese": "Kỹ sư phần mềm"},
    {"english": "Data Analyst", "vietnamese": "Nhà phân tích dữ liệu"}
]
Output: 
- alias_to_english: dictionary ánh xạ mọi tên gọi về bản tiếng Anh chuẩn.
- canonical_english: danh sách các thuật ngữ tiếng Anh chuẩn.
- aliases: danh sách tất cả alias, gồm cả tiếng Anh và tiếng Việt.

map_term_to_englist: Hàm nhận 1 chuỗi text và map nó về tiếng anh nếu có trong lookup (là từ điển)

CSV English/Vietnamese
        ↓
build_bilingual_lookup
        ↓
Tạo lookup:
    tiếng Việt -> tiếng Anh
    tiếng Anh -> tiếng Anh
        ↓
Tạo danh sách alias Anh + Việt
        ↓
Đưa alias vào PhraseMatcher
        ↓
Matcher có thể tìm job title/skill trong văn bản

In [ ]:
def build_bilingual_lookup(entries: list[dict[str, str]]) -> tuple[dict[str, str], list[str], list[str]]:
    alias_to_english = {}
    canonical_english = []

    for item in entries:
        english = clean_line_text_and_lower(item.get("english", ""))
        vietnamese = clean_line_text_and_lower(item.get("vietnamese", ""))

        if not english:
            continue

        canonical_english.append(english)
        alias_to_english[english] = english

        if vietnamese:
            alias_to_english[vietnamese] = english

    canonical_english = sorted(set(canonical_english))
    aliases = sorted(set(alias_to_english.keys()))
    return alias_to_english, canonical_english, aliases


def map_term_to_english(text: str, lookup: dict[str, str] | None = None) -> str:
    normalized = clean_line_text_and_lower(text)

    if not lookup:
        return normalized

    return lookup.get(normalized, normalized)


title_lookup, title_dict, title_aliases = build_bilingual_lookup(title_entries)
skill_lookup, skill_dict, skill_aliases = build_bilingual_lookup(skill_entries)

title_matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
skill_matcher = PhraseMatcher(nlp.vocab, attr="LOWER")

title_matcher.add("JOB_TITLE", [nlp.make_doc(term) for term in title_aliases])
skill_matcher.add("SKILL", [nlp.make_doc(term) for term in skill_aliases])

print("title canonical english:", len(title_dict))
print("title aliases vi+en:", len(title_aliases))
print("skill canonical english:", len(skill_dict))
print("skill aliases vi+en:", len(skill_aliases))


title canonical english: 896
title aliases vi+en: 1783
skill canonical english: 640
skill aliases vi+en: 1252


### 7. Field Extraction

prepare_sections:
- input: Keetsquar trích xuất
{
    "sections": {
        "summary": ["  Data Analyst  ", "", "Data Analyst"],
        "skills": ["Python", "SQL", "Python"],
        "experience": ["Worked at ABC company"]
    }
}
- Lấy phần sextions ra. 
- Nó chạy qua từng Prefered_section_order:
- Nếu kraww_section có 1 sectiom không nằm trong danh sách nyaf nó sẽ bỏ qua
- Duyệt từng dòng trong section:
    - Chuyển dòng đó sang chuỗi và làm sạch
    - Thêm dòng đó vào cleand
- Gán section hiện tại vào dictionary
- Trả về dictionary:
{
    "summary": ["Data Analyst"],
    "skills": ["Python", "SQL"],
    "experience": ["Worked at ABC company"]
}

get_first_match_text: hàm tìm match đầu tiên trong 1 doc spaCy
- doc: văn bản đã được xử lý bằng spaCy, ví dụ doc = nlp("I am a Python developer").
- matcher: PhraseMatcher, ví dụ skill_matcher hoặc title_matcher.
- lookup: dictionary để map alias về tiếng Anh chuẩn, ví dụ "kỹ sư phần mềm" -> "software engineer".
- doc = nlp("I use Python and SQL")
[
    (match_id, 2, 3),
    (match_id, 4, 5)
]



In [37]:
def prepare_sections(extraction_result: dict[str, Any]) -> dict[str, list[str]]:
    raw_sections = extraction_result.get("sections", {})
    prepared = {}

    for section in PREFERRED_SECTION_ORDER:
        lines = raw_sections.get(section, [])
        cleaned = []

        for line in lines:
            line = clean_line_text(str(line))
            if not line:
                continue
            cleaned.append(line)

        prepared[section] = unique_keep_order(cleaned)

    return prepared

def get_first_match_text(doc, matcher, lookup: dict[str, str] | None = None) -> str:
    matches = matcher(doc)
    if not matches:
        return ""

    matches = sorted(matches, key=lambda x: (x[1], -(x[2] - x[1])))
    _, start, end = matches[0]
    return map_term_to_english(doc[start:end].text, lookup)


def get_all_match_texts(doc, matcher, lookup: dict[str, str] | None = None) -> list[str]:
    matches = matcher(doc)
    results = []

    for _, start, end in matches:
        results.append(map_term_to_english(doc[start:end].text, lookup))

    return unique_keep_order(results)


#### Personal info + summary

In [67]:
# Hàm này sẽ trích xuất thông tin cá nhân như tên, email, số điện thoại từ phần header và personal_info
def extract_personal_info(extraction_result: dict[str, Any], sections: dict[str, list[str]]) -> dict[str, str]:
    full_text = extraction_result.get("text", "")
    header_lines = sections.get("header", []) + sections.get("personal_info", [])

    cleaned_text_for_email = re.sub(r'\s+@\s+', '@', full_text)
    cleaned_text_for_email = re.sub(r'\s+\.', '.', cleaned_text_for_email)
    
    email_match = re.search(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b", cleaned_text_for_email)
    
    # 2. Siết chặt Regex Phone (Ưu tiên format VN: 0... hoặc +84...)
    phone_match = re.search(r"(?:(?:\+|00)84|0)[35789][0-9\-\s.]{7,11}\b", full_text)

    email = email_match.group(0).strip() if email_match else ""
    phone = phone_match.group(0).strip() if phone_match else ""

    if phone:
        phone = re.sub(r"[^\d+]", "", phone)

    name = ""
    for line in header_lines[:6]:
        if "@" in line:
            continue
        if re.search(r"\d", line):
            continue
        if len(line.split()) < 2 or len(line.split()) > 6:
            continue
        if detect_section_heading(line):
            continue
        name = line.strip()
        break

    return {
        "name": name,
        "email": email,
        "phone": phone
    }

# Hàm này sẽ trích xuất phần tóm tắt hoặc mục tiêu nghề nghiệp từ phần objective
def extract_summary(sections: dict[str, list[str]]) -> str:
    objective_lines = sections.get("objective", [])
    if objective_lines:
        return " ".join(objective_lines).strip()
    return ""


#### SKILL

clean_skill_output: Chuẩn hóa 1 dòng kĩ năng

extract_skills: Nhận vào extract_result trả về 1 list
- Trả về danh sách skill đã tìm được, được chuẩn hóa về tiếng anh

In [39]:
def clean_skill_output(text: str) -> str:
    text = clean_line_text(str(text)).strip().lower()
    text = re.sub(r"\s+", " ", text)
    return text


def extract_skills(extraction_result: dict[str, Any], sections: dict[str, list[str]]) -> list[str]:
    def should_merge(prev_line: str, curr_line: str) -> bool:
        if not prev_line or not curr_line:
            return False

        prev_line = prev_line.strip()
        curr_line = curr_line.strip()

        if not curr_line:
            return False

        if curr_line[0].islower():
            return True

        if prev_line.count("(") > prev_line.count(")"):
            return True

        if prev_line.endswith((",", "-", "&", "/", ":")):
            return True

        return False

    raw_lines = sections.get("skills", [])
    merged_lines = []

    for raw_line in raw_lines:
        line = clean_skill_output(raw_line)
        if not line:
            continue

        line = re.sub(
            r"^(?:k? n?ng|ky nang|skills?)\s*[:\-]?\s*",
            "",
            line,
            flags=re.IGNORECASE,
        ).strip()

        if not line:
            continue

        if merged_lines and should_merge(merged_lines[-1], line):
            merged_lines[-1] = f"{merged_lines[-1]} {line}".strip()
        else:
            merged_lines.append(line)

    matched_skills = []

    for line in merged_lines:
        doc = nlp(line)
        matched_skills.extend(get_all_match_texts(doc, skill_matcher, skill_lookup))

    if not matched_skills:
        full_text = extraction_result.get("text", "")
        doc = nlp(full_text)
        matched_skills.extend(get_all_match_texts(doc, skill_matcher, skill_lookup))

    return unique_keep_order(matched_skills)


In [96]:
def looks_like_title_line(line: str) -> bool:
    if not line:
        return False

    doc = nlp(line)
    has_title = bool(title_matcher(doc))

    if not has_title:
        return False

    if len(line.split()) > 12:
        return False

    return True

def extract_candidate_titles_from_experience(experience: list[dict[str, str]]) -> list[str]:
    matched_titles = []

    for item in experience:
        raw_title = item.get("job_title", "")
        if not raw_title:
            continue

        doc = nlp(raw_title)
        
        # Lấy tất cả title match được trong từ điển title
        matched_titles.extend(
            get_all_match_texts(doc, title_matcher, title_lookup)
        )

        # Fallback: nếu raw_title đã là alias/canonical exact trong title_lookup
        normalized_title = clean_line_text_and_lower(raw_title)
        if normalized_title in title_lookup:
            matched_titles.append(title_lookup[normalized_title])
    print("Matched titles from experience:", matched_titles)
    return unique_keep_order(matched_titles)


#### Experience helper

In [69]:
def strip_year_range(text: str) -> str:
    text = YEAR_RANGE_RE.sub("", text)
    text = re.sub(r"\s+", " ", text).strip(" -|,;")
    return text.strip()


def split_experience_blocks(lines: list[str]) -> list[list[str]]:
    blocks = []
    current_block = []

    for line in lines:
        line = line.strip()
        if not line:
            continue

        starts_new = False

        if current_block:
            if YEAR_RANGE_RE.search(normalize_for_heading(line)):
                starts_new = True
            elif looks_like_title_line(line):
                starts_new = True

        if starts_new:
            blocks.append(current_block)
            current_block = [line]
        else:
            current_block.append(line)

    if current_block:
        blocks.append(current_block)

    return blocks


def split_blocks_by_title(lines: list[str]) -> list[list[str]]:
    return split_experience_blocks(lines)


In [41]:
def extract_experience(sections: dict[str, list[str]]) -> list[dict[str, str]]:
    lines = sections.get("experience", [])
    if not lines:
        return []

    blocks = split_experience_blocks(lines)
    results = []

    for block in blocks:
        if not block:
            continue

        header_line = block[0]
        doc = nlp(header_line)
        job_title = get_first_match_text(doc, title_matcher, title_lookup)

        if not job_title:
            job_title = clean_line_text_and_lower(strip_year_range(header_line))

        description_lines = block[1:] if len(block) > 1 else []
        description = " ".join(description_lines).strip()

        if job_title or description:
            results.append({
                "job_title": job_title,
                "description": description
            })

    return results


#### Education helper + education

In [42]:
def extract_degree(text: str) -> str:
    norm = normalize_for_heading(text)

    for pattern, label in DEGREE_PATTERNS:
        if re.search(pattern, norm, flags=re.IGNORECASE):
            return label

    return ""

def extract_major_from_block(block: list[str]) -> str:
    block_text = " ".join(block)

    match = MAJOR_REGEX.search(block_text)
    if match:
        major = match.group(1).strip()
        major = re.split(r"\b(?:19|20)\d{2}\b", major, maxsplit=1)[0]
        major = re.split(r"[,;|]", major, maxsplit=1)[0]
        return clean_line_text_and_lower(major)

    for line in block[1:4]:
        line_norm = normalize_for_heading(line)

        if not line_norm:
            continue
        if re.search(r"\b(truong|tr??ng|university|college|hoc vien|h?c vi?n|institute)\b", line_norm):
            continue
        if "gpa" in line_norm:
            continue
        if YEAR_RANGE_RE.search(line_norm):
            continue
        if line.endswith("."):
            continue
        if len(line.split()) > 14:
            continue

        return clean_line_text_and_lower(line)

    return ""

def _looks_like_schoolheading(line: str) -> bool:
    """Đoán xem một dòng có giống tên trường/học viện hay không."""
    if not line:
        return False

    normalized = normalize_for_heading(line)

    if not normalized:
        return False

    # Nếu dòng chứa từ khóa kiểu trường / viện thì rất có khả năng là heading trường
    if SCHOOL_HINT_RE.search(normalized):
        return True

    # Các dòng quá ngắn thường không đủ thông tin để là tên trường
    if len(line.split()) < 2:
        return False

    # Nếu có GPA hoặc dấu câu kết câu thì thường không phải heading trường
    if "gpa" in normalized:
        return False
    if line.strip().endswith("."):
        return False

    # Nếu có khoảng năm thì thường là dòng mốc thời gian, không phải heading trường
    if YEAR_RANGE_RE.search(normalized):
        return False

    # Một số tên trường viết hoa khá mạnh
    if re.match(
        r"^(?:[A-ZÀ-ỸĐ][A-ZÀ-ỸĐa-zà-ỹđ&/()\\-]+\\s+){1,8}[A-ZÀ-ỸĐ][A-ZÀ-ỸĐa-zà-ỹđ&/()\\-]+$",
        line.strip(),
    ):
        if len(line.split()) <= 12:
            return True

    return False


def split_education_blocks(lines: list[str]) -> list[list[str]]:
    blocks = []
    current_block = []

    for line in lines:
        line = line.strip()
        if not line:
            continue

        starts_new = False

        if current_block:
            if _looks_like_schoolheading(line):
                starts_new = True
            elif extract_degree(line):
                starts_new = True

        if starts_new:
            blocks.append(current_block)
            current_block = [line]
        else:
            current_block.append(line)

    if current_block:
        blocks.append(current_block)

    return blocks


def extract_education(sections: dict[str, list[str]]) -> list[dict[str, str]]:
    lines = sections.get("education", [])
    if not lines:
        return []

    blocks = split_education_blocks(lines)
    results = []

    for block in blocks:
        block_text = " ".join(block)
        degree = extract_degree(block_text)
        major = extract_major_from_block(block)

        if not degree and major:
            if len(major.split()) <= 2:
                continue
            if major.endswith("."):
                continue

        if degree or major:
            results.append({
                "degree": degree,
                "major": major,
            })

    return results

#### PROJECT


In [43]:
def extract_projects(sections: dict[str, list[str]]) -> list[dict[str, str]]:
    lines = sections.get("projects", [])
    if not lines:
        return []

    blocks = split_blocks_by_title(lines)
    results = []

    for block in blocks:
        block_text = "\n".join(block)
        doc = nlp(block_text)

        role = get_first_match_text(doc, title_matcher, title_lookup)
        description = " ".join(block).strip()

        if role or description:
            results.append({
                "role": role,
                "description": description
            })

    return results


#### Total years experience

In [44]:
def parse_end_year(token: str) -> int | None:
    import datetime as _dt

    token = normalize_for_heading(token)

    if token in {"present", "current", "nay", "hien tai"}:
        return _dt.date.today().year

    if re.fullmatch(r"(?:19|20)\d{2}", token):
        return int(token)

    return None


def infer_total_years_from_experience(lines: list[str]) -> float | None:
    blocks = split_experience_blocks(lines)
    total_years = 0
    found = False

    for block in blocks:
        header_text = " ".join(block[:2])
        norm = normalize_for_heading(header_text)
        match = YEAR_RANGE_RE.search(norm)

        if not match:
            continue

        start_year = int(match.group(1))
        end_year = parse_end_year(match.group(2))

        if end_year is None:
            continue

        if end_year >= start_year:
            total_years += (end_year - start_year)
            found = True

    return float(total_years) if found else None


def extract_total_years_experience(extraction_result: dict[str, Any], sections: dict[str, list[str]]) -> float | None:
    search_text = normalize_for_heading(
        " ".join(sections.get("objective", [])) + "\n" + extraction_result.get("text", "")
    )

    patterns = [
        r"(\d+(?:\.\d+)?)\s+years?\s+of\s+experience",
        r"over\s+(\d+(?:\.\d+)?)\s+years?",
        r"more than\s+(\d+(?:\.\d+)?)\s+years?",
        r"(\d+(?:\.\d+)?)\s+nam\s+kinh\s+nghiem",
        r"tren\s+(\d+(?:\.\d+)?)\s+nam",
    ]

    for pattern in patterns:
        match = re.search(pattern, search_text, flags=re.IGNORECASE)
        if match:
            return float(match.group(1))

    return infer_total_years_from_experience(sections.get("experience", []))


### 8. Build JSON cuối

#### Build candidate profile

In [93]:
def build_candidate_profile_json(extraction_result: dict[str, Any]) -> dict[str, Any]:
    sections = prepare_sections(extraction_result)
    experience = extract_experience(sections)

    return {
        "candidate_profile": {
            "personal_info": extract_personal_info(extraction_result, sections),
            "summary": extract_summary(sections),
            "skills": extract_skills(extraction_result, sections),
            "titles": extract_candidate_titles_from_experience(experience),
            "experience": experience,
            "education": extract_education(sections),
            "project": extract_projects(sections),
            "total_years_experience": extract_total_years_experience(extraction_result, sections)
        }
    }


#### Pipeline end-to-end cho 1 CV

In [78]:
def build_candidate_profile_text(candidate_json: dict[str, Any]) -> str:
    profile = candidate_json["candidate_profile"]
    chunks = []

    if profile.get("summary"):
        chunks.append(profile["summary"])

    skills = profile.get("skills", [])
    if skills:
        chunks.append(" ".join(skills))
    titles = profile.get("titles", [])
    if titles:
        chunks.append(" ".join(titles))
    for item in profile.get("experience", []):
        if item.get("job_title"):
            chunks.append(item["job_title"])
        if item.get("description"):
            chunks.append(item["description"])

    for item in profile.get("education", []):
        if item.get("degree"):
            chunks.append(item["degree"])
        if item.get("major"):
            chunks.append(item["major"])

    for item in profile.get("project", []):
        if item.get("role"):
            chunks.append(item["role"])
        if item.get("description"):
            chunks.append(item["description"])

    return clean_cv_text("\n".join(chunks))


def process_cv_to_json(pdf_path: str | Path) -> dict[str, Any]:
    pdf_path = Path(pdf_path)

    extraction_result = extract_text_from_pdf(pdf_path)
    cleaned_result = clean_extraction_result(extraction_result)
    candidate_json = build_candidate_profile_json(cleaned_result)
    candidate_profile_text = build_candidate_profile_text(candidate_json)

    return {
        "pdf_path": str(pdf_path),
        "extraction_result": extraction_result,
        "cleaned_result": cleaned_result,
        "candidate_json": candidate_json,
        "candidate_profile_text": candidate_profile_text,
    }


### 9. Debug, batch

In [81]:
def print_extraction_debug(
    extraction_result: dict,
    candidate_json: dict,
    preview_sections: bool = True,
    max_lines_per_section: int = 8,
) -> None:
    profile = candidate_json["candidate_profile"]

    print("=" * 100)
    print("PDF TYPE :", extraction_result.get("pdf_type"))
    print("METHOD   :", extraction_result.get("method"))
    print("WARNINGS :", extraction_result.get("warnings", []))
    print("=" * 100)

    print("\n[PERSONAL INFO]")
    pprint(profile.get("personal_info", {}), sort_dicts=False)

    print("\n[SUMMARY]")
    print(compact_text(profile.get("summary", ""), 300))

    print("\n[SKILLS]")
    print("count =", len(profile.get("skills", [])))
    print(profile.get("skills", []))

    print("\n[TITLES]")
    print("count =", len(profile.get("titles", [])))
    print(profile.get("titles", []))


    print("\n[EXPERIENCE]")
    for i, item in enumerate(profile.get("experience", []), start=1):
        print(f"- experience[{i}]")
        print("  job_title   :", item.get("job_title", ""))
        print("  description :", compact_text(item.get("description", ""), 220))

    print("\n[EDUCATION]")
    for i, item in enumerate(profile.get("education", []), start=1):
        print(f"- education[{i}]")
        print("  degree :", item.get("degree", ""))
        print("  major  :", item.get("major", ""))

    print("\n[PROJECT]")
    for i, item in enumerate(profile.get("project", []), start=1):
        print(f"- project[{i}]")
        print("  role        :", item.get("role", ""))
        print("  description :", compact_text(item.get("description", ""), 220))

    print("\n[TOTAL YEARS EXPERIENCE]")
    print(profile.get("total_years_experience"))

    if preview_sections:
        print("\n" + "=" * 100)
        print("[SECTION PREVIEW]")
        print("=" * 100)

        sections = extraction_result.get("sections", {})
        for section in PREFERRED_SECTION_ORDER:
            lines = sections.get(section, [])
            if not lines:
                continue

            print(f"\n--- {section.upper()} ({len(lines)} lines) ---")
            for line in lines[:max_lines_per_section]:
                print(line)

            if len(lines) > max_lines_per_section:
                print("...")

In [48]:
def flag_candidate_profile(extraction_result: dict, candidate_json: dict) -> list[str]:
    profile = candidate_json["candidate_profile"]
    flags = []

    personal_info = profile.get("personal_info", {})
    skills = profile.get("skills", [])
    experience = profile.get("experience", [])
    education = profile.get("education", [])
    projects = profile.get("project", [])
    total_years = profile.get("total_years_experience")

    if extraction_result.get("pdf_type") == "scan_or_image_based":
        flags.append("scan_pdf_requires_ocr")

    if len(extraction_result.get("text", "")) < 300:
        flags.append("short_extracted_text")

    if not personal_info.get("name"):
        flags.append("missing_name")
    if not personal_info.get("email"):
        flags.append("missing_email")
    if not personal_info.get("phone"):
        flags.append("missing_phone")
    if not skills:
        flags.append("missing_skills")
    if not experience:
        flags.append("missing_experience")
    if not education:
        flags.append("missing_education")

    empty_exp_titles = sum(1 for x in experience if not x.get("job_title"))
    if experience and empty_exp_titles >= max(1, len(experience) // 2):
        flags.append("many_empty_experience_titles")

    empty_project_roles = sum(1 for x in projects if not x.get("role"))
    if projects and empty_project_roles >= max(1, len(projects) // 2):
        flags.append("many_empty_project_roles")

    empty_education = sum(1 for x in education if not x.get("degree") and not x.get("major"))
    if education and empty_education > 0:
        flags.append("empty_education_items")

    if total_years is None:
        flags.append("missing_total_years_experience")

    return flags


In [97]:
pdf_path = Path(r"C:\Users\Admin\Downloads\CV_PTTM.pdf")
processed_result = process_cv_to_json(pdf_path)
flags = flag_candidate_profile(processed_result["cleaned_result"], processed_result["candidate_json"])

print_extraction_debug(
    extraction_result=processed_result["cleaned_result"],
    candidate_json=processed_result["candidate_json"],
    preview_sections=True,
    max_lines_per_section=10,
)

print("\n[FLAGS]")
print(flags)


Matched titles from experience: ['receptionist', 'receptionist']
PDF TYPE : multi_column_or_complex_layout
METHOD   : pymupdf_words_layout_level3
WARNINGS : []

[PERSONAL INFO]
{'name': 'phạm thị trà my',
 'email': 'myphamthitra1805@gmail.com',
 'phone': '0389297446'}

[SUMMARY]
ngắn hạn: mong muốn được làm việc trong môi trường chuyên nghiệp, phù hợp với chuyên ngành đã học, nhằm rèn luyện kỹ năng giao tiếp, tư vấn khách hàng và tích lũy kinh nghiệm thực tế trong lĩnh vực du lịch - lữ hành. dài hạn: phấn đấu nâng cao năng lực chuyên môn, hoàn thiện kỹ năng nghề nghiệp để g...

[SKILLS]
count = 5
['psychology', 'communication', 'counseling', 'client consulting', 'excel']

[TITLES]
count = 1
['receptionist']

[EXPERIENCE]
- experience[1]
  job_title   : receptionist
  description : hường spa đón tiếp khách hàng với thái độ thân thiện, lịch sự và chuyên nghiệp tư vấn các dịch vụ spa phù hợp với nhu cầu của khách hàng giải đáp thắc mắc, hỗ trợ khách hàng trong quá trình sử dụng dịch vụ ti

In [58]:
def run_batch_extraction_debug(folder_path: str | Path, suffixes=(".pdf",)) -> list[dict]:
    folder_path = Path(folder_path)

    if not folder_path.exists():
        raise FileNotFoundError(f"Folder not found: {folder_path}")

    results = []

    for file_path in folder_path.rglob("*"):
        if not file_path.is_file():
            continue
        if file_path.suffix.lower() not in suffixes:
            continue

        try:
            processed_result = process_cv_to_json(file_path)
            flags = flag_candidate_profile(
                processed_result["cleaned_result"],
                processed_result["candidate_json"],
            )

            results.append({
                "file_path": str(file_path),
                "filename": file_path.name,
                "extraction_result": processed_result["extraction_result"],
                "cleaned_result": processed_result["cleaned_result"],
                "candidate_json": processed_result["candidate_json"],
                "candidate_profile_text": processed_result["candidate_profile_text"],
                "flags": flags,
                "error": "",
            })
        except Exception as e:
            results.append({
                "file_path": str(file_path),
                "filename": file_path.name,
                "extraction_result": {},
                "cleaned_result": {},
                "candidate_json": {},
                "candidate_profile_text": "",
                "flags": ["runtime_error"],
                "error": str(e),
            })

    return results


In [59]:
def build_batch_summary_df(batch_results: list[dict]) -> pd.DataFrame:
    rows = []

    for item in batch_results:
        candidate_json = item.get("candidate_json", {})
        profile = candidate_json.get("candidate_profile", {}) if candidate_json else {}
        personal_info = profile.get("personal_info", {}) if profile else {}

        rows.append({
            "filename": item.get("filename", ""),
            "file_path": item.get("file_path", ""),
            "pdf_type": item.get("cleaned_result", {}).get("pdf_type", ""),
            "method": item.get("cleaned_result", {}).get("method", ""),
            "warning_count": len(item.get("cleaned_result", {}).get("warnings", [])),
            "flag_count": len(item.get("flags", [])),
            "flags": ", ".join(item.get("flags", [])),
            "has_name": bool(personal_info.get("name")),
            "has_email": bool(personal_info.get("email")),
            "has_phone": bool(personal_info.get("phone")),
            "skill_count": len(profile.get("skills", [])) if profile else 0,
            "experience_count": len(profile.get("experience", [])) if profile else 0,
            "education_count": len(profile.get("education", [])) if profile else 0,
            "project_count": len(profile.get("project", [])) if profile else 0,
            "total_years_experience": profile.get("total_years_experience") if profile else None,
            "profile_text_length": len(item.get("candidate_profile_text", "")),
            "error": item.get("error", ""),
        })

    df = pd.DataFrame(rows)

    if not df.empty:
        df = df.sort_values(
            by=["flag_count", "warning_count", "filename"],
            ascending=[False, False, True],
        ).reset_index(drop=True)

    return df

def save_candidate_json(candidate_json: dict[str, Any], output_path: str | Path) -> Path:
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(candidate_json, f, ensure_ascii=False, indent=2)

    return output_path


In [60]:
def inspect_batch_item(batch_results: list[dict], filename_keyword: str) -> None:
    matched = [
        item for item in batch_results
        if filename_keyword.lower() in item.get("filename", "").lower()
    ]

    if not matched:
        print("Khong tim thay file phu hop.")
        return

    item = matched[0]

    print("FILE:", item["filename"])
    print("FLAGS:", item["flags"])
    print("ERROR:", item["error"])

    if item["error"]:
        return

    print_extraction_debug(
        extraction_result=item["cleaned_result"],
        candidate_json=item["candidate_json"],
        preview_sections=True,
        max_lines_per_section=12,
    )


In [61]:
def export_candidate_json_folder(
    folder_path: str | Path,
    output_dir: str | Path,
    suffixes=(".pdf",),
    include_profile_text: bool = True,
) -> pd.DataFrame:
    folder_path = Path(folder_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    rows = []

    for file_path in folder_path.rglob("*"):
        if not file_path.is_file():
            continue
        if file_path.suffix.lower() not in suffixes:
            continue

        processed_result = process_cv_to_json(file_path)
        candidate_json = processed_result["candidate_json"]
        candidate_profile_text = processed_result["candidate_profile_text"]

        json_path = output_dir / f"{file_path.stem}.json"
        save_candidate_json(candidate_json, json_path)

        profile_text_path = None
        if include_profile_text:
            profile_text_path = output_dir / f"{file_path.stem}_profile_text.txt"
            profile_text_path.write_text(candidate_profile_text, encoding="utf-8")

        rows.append({
            "filename": file_path.name,
            "json_path": str(json_path),
            "profile_text_path": str(profile_text_path) if profile_text_path else "",
            "skill_count": len(candidate_json["candidate_profile"].get("skills", [])),
            "experience_count": len(candidate_json["candidate_profile"].get("experience", [])),
            "education_count": len(candidate_json["candidate_profile"].get("education", [])),
            "profile_text_length": len(candidate_profile_text),
        })

    manifest_df = pd.DataFrame(rows)
    manifest_path = output_dir / "candidate_export_manifest.csv"
    manifest_df.to_csv(manifest_path, index=False, encoding="utf-8")

    print("Da luu manifest vao:")
    print(manifest_path)

    return manifest_df
